#                                            DIRECTIONAL DRILLING 
##                                           *DIRECTIONAL WELLS PROFILES AND DIRECTIONAL WELLS TRAJECTORIES*

***

![well](Resources/Well_prof.jpg)

# Python Libraries

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import namedtuple
from math import radians, isclose, acos, asin, cos, sin, tan, atan, degrees, sqrt

# *Directional Wells Profiles*

## *Slant Well Profile (J Type)*

![j](Resources/j_prof.png)

In [3]:
Data = namedtuple("Input", "TVD KOP BUR DH")
Output = namedtuple("Output", "R Theta TVD_EOB Md_EOB Dh_EOB Tan_len Md_total")

def well_J(data:Data, unit='ingles') -> Output:
    tvd = data.TVD
    kop = data.KOP
    bur = data.BUR
    dh = data.DH
    if unit == 'ingles':
        R = 5729.58 / bur
    else:
        R = 1718.87 / bur

    # segment DC
    if dh > R:
        dc = dh - R
    elif dh < R:
        dc =   R - dh

    # Segment DO
    do = tvd - kop

    # DOC Angle
    doc = degrees(atan(dc / do))

    # Segment OC
    oc = sqrt(dc**2 + do**2)

    #BOC Angle
    boc = degrees(acos(R / oc))

    # BOD Angle
    if R < dh:
        bod = boc - doc
    elif R > dh:
        bod = boc + doc

    # Theta angle
    theta = 90 - bod

    # TVD a EOB
    tvd_eob = kop + abs(R * sin(radians(theta)))

    # MD a EOB
    if unit == 'ingles':
        md_eob = kop + (theta / bur) * 100
    else:
        md_eob = kop + (theta / bur) * 30

    # DH a EOB
    dh_eob = R - R * cos(radians(theta))

    # Tangent section
    tan_len = sqrt(oc**2 - R**2)

    # MD Total
    if unit == 'ingles':
        md_total = kop + (theta / bur) * 100 + tan_len
    else:
        md_total = kop + (theta / bur) * 30 + tan_len

        
    return Output(R=R, Theta=theta, TVD_EOB=tvd_eob, Md_EOB=md_eob, Dh_EOB=dh_eob, \
                  Tan_len=tan_len, Md_total=md_total)

## *Ejercicio 1*

In [4]:
# data
tvd = 8000 #ft
kop = 500 #ft
bur = 2 #o/100ft
dh = 970.8 #ft

In [5]:
trajectory_J = well_J(Data(tvd, kop, bur, dh))
trajectory_J

Output(R=2864.79, Theta=7.564230623470863, TVD_EOB=877.1139517978513, Md_EOB=878.2115311735431, Dh_EOB=24.929649303260703, Tan_len=7185.414140882904, Md_total=8063.625672056447)

In [6]:
names = ['R', 'theta', 'tvd_EOB', 'Md_EOB', 'Dh_EOB', 'Lengh_tan', 'Md_Total']
for param, value in zip(names, trajectory_J):
    if param == 'theta':
        print(f"{param} : {value:.3f} degrees")
    else:
        print(f"{param} : {value:.3f} ft")

R : 2864.790 ft
theta : 7.564 degrees
tvd_EOB : 877.114 ft
Md_EOB : 878.212 ft
Dh_EOB : 24.930 ft
Lengh_tan : 7185.414 ft
Md_Total : 8063.626 ft


## *S-Type Well Profile*

![s](Resources/s_prof.png)

In [7]:
# Function for S-Type wells

# Function to calculate parameters from a S-Type well
Data_S = namedtuple("Input", "TVD KOP BUR DOR DH")
Output_S = namedtuple("Output", "R1 R2 Theta TVD_EOB Md_EOB Dh_EOB Tan_len Md_SOD TVD_SOD Dh_SOD Md_total")


# *Ejercicio 2*

In [8]:
# Data
kop = 6084 #ft
tvd = 12000 #ft
bur = 3 #o/100ft
dor = 2 #o/ft
dh = 3500 #ft

In [9]:
Data = namedtuple("Input", "TVD KOP BUR DOR DH")
Output = namedtuple("Output", "R Theta TVD_EOB Md_EOB Dh_EOB Tan_len Md_total")

def Pozo_tipo_S_ft(data:Data, unit='ingles') -> Output:

    tvd = data.TVD
    kop = data.KOP
    bur = data.BUR
    dor = data.DOR
    dh = data.DH

    if unit == 'ingles':
        R1 = 5729.58 / bur
    else:
        R1 = 1718.87 / bur

    if unit == 'ingles':
        R2 = 5729.58 / dor
    else:
        R2 = 1718.87 / dor


    if dh > R1 + R2:
        FE = dh - (R1 + R2)
    if dh < R1 + R2:
        FE = R1 - (dh - R2)

    EO = tvd - kop

    A_FOE = degrees(atan(FE/EO))

    OF = sqrt(FE**2 + EO**2)

    FG = R1 + R2

    A_FOG = degrees(asin(FG/OF))

    theta = A_FOG - A_FOE                           #Maximo angulo de construccion

    TVD_EOB = kop + abs(R1 * sin(radians(theta)))

    # MD a EOB
    if unit == 'ingles':
        md_eob = kop + (theta / bur) * 100
    else:
        md_eob = kop + (theta / bur) * 30


    dh_eob = R1 - abs(R1 * cos(radians(theta)))            #D1 = distancia horizontal hasta end of build

    BC = sqrt(OF**2 - FG**2)

    MD_SOD = md_eob + BC                            #MD hasta start of drop

    TVD_SOD = TVD_EOB + abs(BC * cos(radians(theta)))           #TVD_V3 = (V3) = TVD hasta start of drop

    DH_SOD = dh_eob + abs(BC * sin(radians(theta)))            #D2 = distancia horizontal hasat start of drop


    # MD Total
    if unit == 'ingles':
        MDT = MD_SOD + (theta/dor)*100
    else:
        MDT = MD_SOD + (theta/dor)*30

    return Output(R1=R1, R2=R2, Theta=theta, TVD_EOB=TVD_EOB, Md_EOB=md_eob, Dh_EOB=dh_eob, \
                  Tan_len=tan_len, Md_total=md_total)



## *Horizontal Well Profiles*

![hor](Resources/Horizontal_prof.jpg)

In [10]:
# Function for horizontal wells

# Function to calculate parameters of a Horizontal Well
Data_H = namedtuple("Input", "TVD KOP BUR1 BUR2 DH")
Output_H = namedtuple("Output", "R1 R2 Theta TVD_EOB1 Md_EOB1 Dh_EOB1 Tan_len Md_SOB2 Md_total")



## *Ejercicio 3*

In [11]:
# Data
tvd = 3800 #ft
kop = 2000 #ft
bur1 = 5.73 #o/100ft
bur2 = 9.55 #o/100ft
dh = 1800 #ft

***